# Patent Landscaping — Snorkel vs MAS (Bergeaud pool)

Candidate pool = **Bergeaud & Verluise official Self-Driving-Vehicle CPC ∩ keyword** (the
gold-set authors' own search query) merged with the seed pool → 11,196 autonomous patents.
Both arms label the full set (`candidate_all`, OOD included), but the **no-OOD** experiment
trains on the **autonomous pool only (`--ood-n 0`)** — Bergeaud's ADAS-rich pool already supplies
abundant in-domain hard negatives, so the out-of-domain crutch can be dropped.

**Two experiments, four models** (existing no-OOD kept as-is, OOD added on top):
- `snorkel_noood`, `mas_noood` — autonomous pool only, no OOD negatives.
- `snorkel_ood`, `mas_ood` — same pool **+ all OOD negatives** (5 other domains, ~6.3k rows).
  Lets us see whether adding the out-of-domain crutch helps or hurts on the gold set.

Runtime → GPU (T4/L4) → Run all.
Fresh session: run cells 1–4, then cell 5 restores the models from Drive (no retrain).

In [ ]:
# 1) setup
import os
%cd /content
REPO = 'https://github.com/PassionChicken-Leesuin/Patent_Landscaping_Final.git'
if not os.path.exists('/content/Patent_Landscaping_Final'):
    !git clone $REPO
%cd /content/Patent_Landscaping_Final
!git pull
!pip -q install snorkel transformers datasets scikit-learn accelerate

In [ ]:
# 2) mount Drive (persistent model store)
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/patent_landscaping_outputs'
os.makedirs(DRIVE, exist_ok=True); os.makedirs('outputs', exist_ok=True)
print('Drive store:', DRIVE)

In [ ]:
# 3) data prep + build the unified candidate set (Bergeaud pool + OOD)
!python -m scripts.build_dataset
!python -m scripts.build_candidate_all

In [ ]:
# 4) SNORKEL — label the FULL candidate set (autonomous pool + OOD together)
!python -m scripts.run_snorkel --input DataSet/processed/candidate_all.csv

MAS already labeled `candidate_all` (committed `mas_ranked_scores.csv`) — no API call here.

In [ ]:
# 5) train each arm.  Restore from Drive if present.
#   *_noood = autonomous pool only (--ood-n 0);  *_ood = pool + all OOD negatives.
import shutil
FORCE = False
EPOCHS, MAX_LEN = 4, 256

def ensure(name, run_args):
    local, dstore = f'outputs/scibert_{name}', f'{DRIVE}/scibert_{name}'
    if not FORCE and os.path.isfile(f'{local}/config.json'):
        if not os.path.isfile(f'{dstore}/config.json'):
            shutil.copytree(local, dstore, dirs_exist_ok=True)
            if os.path.isfile(f'outputs/metrics_{name}.json'): shutil.copy(f'outputs/metrics_{name}.json', DRIVE)
        print(f'[{name}] local'); return
    if not FORCE and os.path.isfile(f'{dstore}/config.json'):
        shutil.copytree(dstore, local, dirs_exist_ok=True)
        if os.path.isfile(f'{DRIVE}/metrics_{name}.json'): shutil.copy(f'{DRIVE}/metrics_{name}.json', 'outputs/')
        print(f'[{name}] restored'); return
    print(f'[{name}] training (~30 min) ...')
    get_ipython().system(f'python -m scripts.run_downstream {run_args} --epochs {EPOCHS} --max-len {MAX_LEN}')
    shutil.copytree(local, dstore, dirs_exist_ok=True)
    if os.path.isfile(f'outputs/metrics_{name}.json'): shutil.copy(f'outputs/metrics_{name}.json', DRIVE)
    print(f'[{name}] done + saved')

# no-OOD (existing)
ensure('snorkel_noood', '--arm snorkel --unified --ood-n 0 --tag noood')
ensure('mas_noood',     '--arm mas --unified --ood-n 0 --tag noood')
# + OOD (all out-of-domain negatives; omit --ood-n -> include them all)
ensure('snorkel_ood',   '--arm snorkel --unified --tag ood')
ensure('mas_ood',       '--arm mas --unified --tag ood')

In [ ]:
# 6) train/val curves
from src.downstream.plots import plot_history
for n in ['snorkel_noood', 'mas_noood', 'snorkel_ood', 'mas_ood']:
    try: plot_history(n)
    except FileNotFoundError: print(f'[{n}] no history')

In [ ]:
# 7) compare @0.5.  acc=(TP+TN)/all.  auc_seed_vs_hard = automate-vs-assist (the crux).
import json
def show(name):
    p = f'outputs/metrics_{name}.json'
    if not os.path.isfile(p): print(f'{name:14s} (missing)'); return
    r = json.load(open(p))
    seed = r['by_expansion_level'].get('SEED', {}).get('recall(TP rate)', float('nan'))
    print(f"{name:14s} acc={r['accuracy']:.3f} F1={r['macro_f1']:.3f} AUC={r['auc']:.3f} "
          f"AUC(vs hard)={r.get('auc_seed_vs_hard', float('nan')):.3f} "
          f"AUC(vs easy)={r.get('auc_seed_vs_easy', float('nan')):.3f} "
          f"R={r['recall']:.3f} P={r['precision']:.3f} SEED-recall={seed:.3f}")
for n in ['snorkel_noood', 'mas_noood', 'snorkel_ood', 'mas_ood']:
    show(n)

## 8) Threshold calibration + sweep (no retrain)
Tunes the threshold on validation (never gold) and shows the full precision/recall tradeoff.

In [ ]:
!python -m scripts.calibrate_eval
!python -m scripts.threshold_sweep